# Capstone Project – Neuromorphic Locomotion Control for a Quadruped Robot in MuJoCo using a Central Pattern Generator


In the following notebook, we implement a so called "ant"-model which navigates a maze by using a Central Pattern Generator for its forward motion. Sensory Feedback is added to identify the obstacle it encountered, enabling the agnet to select the best strategy to overcome it. The environment contains a simple maze, a wall obstructing the agents desired path and uneven terrain to show the walking gait's robustness. Due to unforseen problems with the climbing strategy, we decided to work around that problem for now, but still left the beginnings of the climbing controller in the notebook as a rough idea of how it might work.

To run the notebook, the files ant_final.xml and terrain.png need to be in the same folder. 

In [1]:
#import of needed libraries
import mujoco
import mediapy as media
import numpy as np
from pathlib import Path


In [2]:
# Model setup
XML_PATH = "ant_FINAL.xml"

if not Path(XML_PATH).exists():
    raise FileNotFoundError(f"Could not find {XML_PATH}. Put ant_presentation.xml in the same folder as this notebook.")

model = mujoco.MjModel.from_xml_path(XML_PATH)

print("Number of actuators:", model.nu)
print("qpos value:", model.nq)
print("qvel value:", model.nv)


Number of actuators: 8
qpos value: 15
qvel value: 14


The next section outputs the sensor namens and dimensions to ensure that they are located correctly and no two sensors share the same position. This problem occured before, leading to incorrect sensor outputs, which then inhibited any reaction of the "ant" to the obstacle in front of it. 

In [3]:
for i in range(model.nsensor):
    print(
        i,
        model.sensor(i).name,
        model.sensor_dim[i],
        model.sensor_adr[i],
    )

0 torso_gyro 3 0
1 torso_acc 3 3
2 front_distance 1 6
3 left_side_distance 1 7
4 right_side_distance 1 8
5 ground_distance 1 9
6 front_height_sensor 1 10
7 rear_distance 1 11
8 front_left_20 1 12
9 front_right_20 1 13
10 front_left_55 1 14
11 front_right_55 1 15
12 foot_1_touch 1 16
13 foot_1_front_touch 1 17
14 foot_2_touch 1 18
15 foot_2_front_touch 1 19
16 foot_3_touch 1 20
17 foot_3_front_touch 1 21
18 foot_4_touch 1 22
19 foot_4_front_touch 1 23


In [4]:
"""
Leg mapping

Each leg stores:
- actuator ids for hip and ankle motor 
- qpos ids for hip and ankle joint position 
- qvel ids for hip and ankle joint velocity
- signs to mirror the same target motion to each side
"""
legs = {
    "back_right": {
        "actuators": (0, 1),   # hip_4, ankle_4
        "qpos": (13, 14),
        "qvel": (12, 13),
        "ankle_sign": 1,
        "hip_sign": -1,
        "phase_offset": 0.00,
        "hip_scale": 1.0,
        "ankle_scale": 1.0,
    },
    "front_left": {
        "actuators": (2, 3),   # hip_1, ankle_1
        "qpos": (7, 8),
        "qvel": (6, 7),
        "ankle_sign": 1,
        "hip_sign": 1,
        "phase_offset": 0.50,
        "hip_scale": 1.0,
        "ankle_scale": 1.0,
    },
    "front_right": {
        "actuators": (4, 5),   # hip_2, ankle_2
        "qpos": (9, 10),
        "qvel": (8, 9),
        "ankle_sign": -1,
        "hip_sign": -1,
        "phase_offset": 0.25,
        "hip_scale": 1.0,
        "ankle_scale": 1.0,
    },
    "back_left": {
        "actuators": (6, 7),   # hip_3, ankle_3
        "qpos": (11, 12),
        "qvel": (10, 11),
        "ankle_sign": -1,
        "hip_sign": 1,
        "phase_offset": 0.75,
        "hip_scale": 1.0,
        "ankle_scale": 1.0,
}}

In [5]:
sensor_names = [
    "torso_gyro",
    "torso_acc",
    "front_distance",
    "left_side_distance",
    "right_side_distance",
    "rear_distance",
    "front_height_sensor",
    "ground_distance",
    "foot_1_touch",
    "foot_1_front_touch",
    "foot_2_touch",
    "foot_3_touch",
    "foot_4_touch",
]

for name in sensor_names:
    sid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SENSOR, name)
    if sid == -1:
        print(name, "NOT PRESENT IN THE MODEL")
        continue
    adr = model.sensor_adr[sid]
    dim = model.sensor_dim[sid]
    print(name, "adr:", adr, "dim:", dim)


torso_gyro adr: 0 dim: 3
torso_acc adr: 3 dim: 3
front_distance adr: 6 dim: 1
left_side_distance adr: 7 dim: 1
right_side_distance adr: 8 dim: 1
rear_distance adr: 11 dim: 1
front_height_sensor adr: 10 dim: 1
ground_distance adr: 9 dim: 1
foot_1_touch adr: 16 dim: 1
foot_1_front_touch adr: 17 dim: 1
foot_2_touch adr: 18 dim: 1
foot_3_touch adr: 20 dim: 1
foot_4_touch adr: 22 dim: 1


We can see, that every sensor has a different adress and only the torso-sensors are 3-dimensional. Therefore, the sensor mappings are correct and should work as intended in the decision-making controller.

To have all parameters in one space, we put them in a single code-block, ensuring easy access and an ordered structure.

In [6]:

obstacle_mode = False  # there is no obstacle in the beginning, so the robot can walk normally

WALK_DIRECTION = 1.0    # Multiplies the hip direction -> ant walks "right" towards the +x direction


# Phase parameters

FREQUENCY = 0.87    # How fast the gait cycle runs (Higher -> faster steps, Lower -> slower and more stable steps)

SWING_END = 0.28    # Fraction of the gait cycle used for swing phase (smaller = quicker swing, more time on the ground)

HOLD_END = 0.48     # End of the short touchdown/hold phase after swing (larger = leg stays forward longer before stance sweep begins)


# Hip target parameters

HIP_BACK = -0.28    # Hip target at the back/end of stance (more negative legs sweeps farther backward, more propulsion)

HIP_FRONT = 0.50    # Hip target at the front/end of swing (larger positive = leg reaches farther forward)

HIP_STANCE = 0.38   # Hip target during the front hold / touchdown phase (higher = keeps the foot more forward after swing)


# Ankle target parameters

ANKLE_LIFT_FRONT = 0.42     # Ankle target for front legs during swing (smaller = retract the leg more)

ANKLE_LIFT_BACK = 0.34      # Ankle target for back legs during swing (smaller = more lift for hind legs)

ANKLE_SUPPORT = 0.78    # Ankle target during touchdown and stance (Higher = leg extends/presses more strongly into the ground)


# PD controller parameters

KP_HIP = 5.0    # Hip proportional gain.

KD_HIP = 0.9    # Hip damping gain (Higher = reduces oscillation and overshoot)

KP_ANKLE = 6.5  # Ankle proportional gain (Higher = ankle presses/retracts more decisively)

KD_ANKLE = 1.1  # Ankle damping gain (Higher = smoother ankle motion)


# Constants for navigating around walls (+ maze)
# These are not guesses, rather each one was measured against the model. The comment on each says which measurement pins it down, so they can be re-derived if the gait or the body ever changes

WALL_FOLLOW_TARGET = 1.60   # Gap the ant tries to hold between its torso centre and the right-hand wall
"""
The feet reach 0.88 m sideways, and a left pivot slides the body 0.51 m *toward* the wall while it turns. 
Anything below ~1.5 m therefore guarantees the right legs catch the wall at the next corner
"""

WALL_LOST_DISTANCE = 2.70   # Above this the right wall counts as gone -> an opening, or the end of a wall

FRONT_BLOCK_DISTANCE = 1.60     # Front reading that counts as "blocked"

CORNER_CLEAR_DISTANCE = 1.20    # How far to keep walking after the right wall disappears, before pivoting right

REACQUIRE_MAX_DISTANCE = 3.50      # After a right turn, how far to walk looking for the wall again before giving up



# steering
HEADING_TOLERANCE = np.radians(5.0)   # dead zone, so the ant does not chatter when aligned

STEER_DEADBAND = 0.30   # Measured: any |steer| below ~0.25 produces no yaw rate whatsoever -> every non-zero correction gets lifted to at least this magnitude

STEER_GAIN = 1.10          # extra steer per radian of heading error
STEER_MAX_FOLLOW = 0.55    # cap while walking, so the ant keeps making forward progress

WALL_STEER_GAIN = 1.00  # Metres of wall-distance error -> radians of heading offset

WALL_STEER_MAX = np.radians(28.0)


# pivot turns 
PIVOT_TOLERANCE = np.radians(7.0)
PIVOT_SETTLE_TIME = 0.25   # yaw must hold on target this long before the turn counts
PIVOT_TIMEOUT = 14.0       # longer than this means the ant is not turning


# sensor conditioning 
SIDE_SENSOR_OFFSET = 0.30   # The left/right sites sit 0.3 m off the torso centreline, so the raw reading is 0.3 m short of the torso-to-wall distance the controller reasons about

MAX_YAW_CORRECTION = np.radians(30.0)

ALIGN_TOLERANCE = np.radians(18.0)  # Side readings are only believed while the ant is this well aligned with the corridor

DEBOUNCE_TIME = 0.20       # a trigger must persist this long before it switches state


# getting unstuck 
STUCK_WINDOW = 6.0       
STUCK_DISTANCE = 0.25      
BACKUP_TIME = 1.6         


# Telling the surroundings apart and rounding a free-standing obstacle

OPEN_SKY_DISTANCE = 10.0
OPEN_SKY_WINDOW = 1.0
OPEN_SKY_FRACTION = 0.5
OPEN_SKY_RELEASE = 0.15

OPEN_SKY_PENDING = 0.15     # "Neither side sensor has seen anything within 10 m for most of the last 2 s" means the ant is not in a corridor any more

WALK_HOME_HEADING = 0.0     # The direction the ant is ultimately trying to travel

WALL_SCAN_DISTANCE = 6.0    # Start looking for the shorter way round as soon as something is this close

ANGLED_CLEAR_MARGIN = 1.0
ANGLED_ALIGN_TOLERANCE = np.radians(6.0)
ANGLED_AGREE_TIME = 0.15

WALL_CORNER_CLEAR = 2.00    # The free-standing wall needs a wider swing round its corners than the labyrinth does

WALL_SIDE_LOST = 3.40   # Same idea as WALL_LOST_DISTANCE (2.70) but for the free-standing detour

WALL_GONE_HOLD = 0.60   # How long "the wall on my side has ended" must hold before the detour believes it

WALL_RETURN_TOLERANCE = 0.40    # How close to the original line of travel counts as "back where I started", which is what ends the detour

WALL_MAX_LEG_DISTANCE = 14.0    # Safety stop: if one face runs longer than this, the obstacle was not free-standing after all, so hand back to maze following rather than walk around it forever


# Crossing sloped / uneven ground 

TERRAIN_FREQUENCY = 0.55    # Gait cycle used while crossing terrain, against 0.87 on the flat 
"""
This one number is the whole terrain adaptation, and it was the only thing that helped: 
    at 0.87 the ant barely gets up the entry ramp (3.5 m in 120 s), 
    at 0.55 it crosses the entire field and keeps going, 100% upright. 
    -> A slower cycle gives each foot longer to find purchase before the body is asked to move over it.

Reflex schemes were tried first and all of them made it *worse*: ground-search (reach further down when a stance foot has no contact), body-lift, and roll/pitch
attitude push together cut the crossing rate from 0.071 to 0.028 m/s. The extra ankle extension they command destabilises a gait that is already coping. 
They are not here because they were measured and rejected, not because they were overlooked.
"""
TERRAIN_SLOPE_WINDOW = 1.5   # seconds of pitch averaging used to spot a slope
TERRAIN_ENTER_SLOPE = 3.0   # Degrees of sustained nose-up needed to decide the ground is rising

TERRAIN_ENTER_HOLD = 0.8     # and it must persist this long, so one lurch cannot trigger it

TERRAIN_STUCK_WINDOW = 25.0
TERRAIN_STUCK_DISTANCE = 0.35 
"""
The ordinary stuck test (0.25 m in 6 s) is far too impatient for terrain. 
Crossing hills the ant genuinely makes only ~0.03 m/s, i.e. 0.18 m in 6 s, so the test fired on healthy progress, backed the ant up, and destroyed the ground it had just gained
a deadlock entirely of the controller's own making. On terrain it has to be slow progress that counts as progress.
"""
REALIGN_TIMEOUT = 24.0

LANE_STEER_GAIN = 0.45
LANE_STEER_MAX = np.radians(25.0)

# Creating and updating the CPG's Oscillator-States
The next block defines the underlying system for the CPG-Controller. It initializes a coupling strength of 2.0 to have faster phase corrections between the four oscillators, in other words, couple the oscillators together. With the make_cpg_state() function we create a starting state for each oscillator based on the pre-defined leg mappings. To advance the oscillator phases over one time step we added the update_cpg_phases(...) function. The function copies the current phases and converts the gait frequency into angular values. For each leg, it compares its current phase to the other legs and calculates how far it is from the desired phase relation. The correction term uses the previously calculated phase error to adjust the leg's movement so all four legs stay coordinated. Lastly, the updated phase gets mapped back to the phase cycle ensuring smooth, rhythmic movement over time.

In [7]:
CPG_COUPLING_STRENGTH = 2.0     # How strongly the oscillators correct deviations from their desired phase relationships

def make_cpg_state():
    """
    Create one persistent oscillator phase for every leg.
    """
    return {
        "phases": {
            leg_name: 2.0 * np.pi * leg.get("phase_offset", 0.0)
            for leg_name, leg in legs.items()
        }
    }


def update_cpg_phases(cpg_state, dt, frequency=FREQUENCY, coupling_strength=CPG_COUPLING_STRENGTH):
    """
    Advance the four coupled leg oscillators by one simulation step.
    The coupling term restores the desired phase relationships if one oscillator is disturbed.
    """
    phases = cpg_state["phases"]

    # Use a copy so all oscillators are updated from the same old state
    old_phases = phases.copy()

    # Convert cycles per second into radians per second
    omega = 2.0 * np.pi * frequency

    number_of_other_legs = max(len(old_phases) - 1, 1)
    new_phases = {}

    for leg_i, theta_i in old_phases.items():
        coupling_correction = 0.0

        offset_i = legs[leg_i].get("phase_offset", 0.0)

        for leg_j, theta_j in old_phases.items():
            if leg_i == leg_j:
                continue

            offset_j = legs[leg_j].get("phase_offset", 0.0)

            # Desired phase of j relative to i
            desired_difference = (2.0 * np.pi * (offset_j - offset_i))

            phase_error = (theta_j - theta_i - desired_difference)

            coupling_correction += np.sin(phase_error)

        # Divide by the number of other oscillators so the total coupling does not become excessively large
        coupling_correction /= number_of_other_legs

        phase_velocity = (omega + coupling_strength * coupling_correction)

        new_phases[leg_i] = (theta_i + phase_velocity * dt) % (2.0 * np.pi)

    phases.update(new_phases)

# Movement State and Obstacle-Handling Logic
In the next section we define the movement state of the ant, including its current behavior mode and internal variables for managing climbing and maze navigation. As mentioned before, the climbing functions do not work yet and are therfore commented-out, but left in the code to possibly work on them further in the future. 
 

In [8]:
CLIMB_SEQUENCE = ["front_left", "front_right", "back_left", "back_right"]

FOOT_TOUCH_SENSOR = {
    "front_left": "foot_1_touch", "front_right": "foot_2_touch",
    "back_left": "foot_3_touch", "back_right": "foot_4_touch",
}
FOOT_FRONT_SENSOR = {
    "front_left": "foot_1_front_touch", "front_right": "foot_2_front_touch",
    "back_left": "foot_3_front_touch", "back_right": "foot_4_front_touch",
}
FOOT_SITE = {
    "front_left": "foot_1_touch_site", "front_right": "foot_2_touch_site",
    "back_left": "foot_3_touch_site", "back_right": "foot_4_touch_site",
}

# Rough resting foot height, used only to tell "on the ground" apart from "on top of something" 
GROUND_FOOT_Z_MARGIN = 0.06  # a foot this much above ground level counts as "elevated"


def make_movement_state():
    # creates a structured state dictionary that stores the ant’s current mode, timing information, planted legs, and other control data
    return {
        "mode": "walk",
        "mode_start": 0.0,

        # box-climbing sub-state (states 3-8)
        "active_leg": None,
        "probe_queue": list(CLIMB_SEQUENCE),
        "planted_legs": set(),
        "locked_joint_targets": {},
        "estimated_obstacle_height": None,
        "ground_foot_z": None,
        "pre_climb_torso_z": None,
        "descend_settled_since": None,
        "box_cleared": False,

        # labyrinth sub-state
        "desired_heading": 0.0,   # the cardinal direction the ant is currently committed to
        "anchor": None,           # where the current mode started, for distance tracking
        "pivot_target": None,
        "pivot_reached_at": None,
        "resume_after_backup": "maze_reacquire_right_wall",
        "after_pivot": "maze_reacquire_right_wall",
        "right_belief": np.inf,   # last trustworthy right-wall distance
        "left_belief": np.inf,    # same for the left, used by the free-standing detour
        "cond_since": {},         # debounce timers, one per trigger
        "prog_anchor": np.zeros(2),
        "prog_time": 0.0,

        # free-standing-obstacle sub-state
        "wall_side": +1,          # which hand the obstacle is on once the ant turned
        "wall_corners_turned": 0,
        "wall_start_pos": None,
        "wall_start_heading": 0.0,
        "go_around_side": 0,      # latched during the approach: +1 left, -1 right
        "open_sky_level": 0.0,    # persistent: a fact about the surroundings, not the mode
        "open_sky": False,        # latched with hysteresis from open_sky_level
        "slope_level": 0.0,       # persistent: running nose-up angle, +ve = climbing
        "home_y": 0.0,            # the lane the ant travels in once out in the open

        # cross-check only, not used for branching
        "obstacle_type": None,
    }


'''
The following functions were generated for the climbing controller
'''

"""
def neutral_stance_target(leg_name, leg):
    #All-legs-down bracing pose used before/between climbing steps
    hip_scale = leg.get("hip_scale", 1.0)
    ankle_scale = leg.get("ankle_scale", 1.0)
    desired_hip = 0.15 * WALK_DIRECTION * leg["hip_sign"] * hip_scale
    desired_ankle = 0.82 * leg["ankle_sign"] * ankle_scale
    return desired_hip, desired_ankle


def get_foot_height(data, model, leg_name):
    sid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, FOOT_SITE[leg_name])
    return data.site_xpos[sid][2]


def get_foot_position(data, model, leg_name):
    sid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, FOOT_SITE[leg_name])
    return data.site_xpos[sid].copy()


def foot_has_contact(data, model, leg_name):
    return get_sensor(data, model, FOOT_TOUCH_SENSOR[leg_name])[0] > 0.01


def foot_has_front_contact(data, model, leg_name):
    return get_sensor(data, model, FOOT_FRONT_SENSOR[leg_name])[0] > 0.01


def foot_is_elevated(data, model, leg_name, movement_state):
    if movement_state["ground_foot_z"] is None:
        return False
    z = get_foot_height(data, model, leg_name)
    contact = foot_has_contact(data, model, leg_name) or foot_has_front_contact(data, model, leg_name)
    return contact and (z - movement_state["ground_foot_z"]) > GROUND_FOOT_Z_MARGIN


def lock_leg_target(data, model, movement_state, leg_name):
    # Lock the current joint positions so that the leg holds its current position once it is planted
    leg = legs[leg_name]
    hip_qpos, ankle_qpos = leg["qpos"]
    movement_state["locked_joint_targets"][leg_name] = (data.qpos[hip_qpos], data.qpos[ankle_qpos],)
"""


'''
The following functions are used for the maze navigation algorithm
'''

def get_yaw(data):        #the orientation of the ant
    w, x, y, z = data.qpos[3:7]
    return np.arctan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))


def get_roll_pitch(data):
    """Roll and pitch of the torso, needed to un-tilt the rangefinder readings"""
    w, x, y, z = data.qpos[3:7]
    roll = np.arctan2(2 * (w * x + y * z), 1 - 2 * (x * x + y * y))
    pitch = np.arcsin(np.clip(2 * (w * y - z * x), -1, 1))
    return roll, pitch

def read_range(data, model, name):
    """Raw rangefinder value, with a miss (-1) turned into +infinity"""
    value = float(get_sensor(data, model, name)[0])
    return np.inf if value < 0 else value


def cos_clamped(angle):
    """This clamp was used to takle the problem of the ant spiraling into permanent spinning, without even encountering an obstacle"""
    return float(np.cos(np.clip(angle, -MAX_YAW_CORRECTION, MAX_YAW_CORRECTION)))


def stable_side_distance(data, model, desired_heading, side=+1):
    """Side distance, projected onto the corridor axis"""
    yaw_error = angle_diff(get_yaw(data), desired_heading)
    if abs(yaw_error) > ALIGN_TOLERANCE:
        return None                      
    raw = read_range(data, model, "right_side_distance" if side > 0 else "left_side_distance")
    if not np.isfinite(raw):
        return np.inf
    roll, _ = get_roll_pitch(data)
    return raw * cos_clamped(yaw_error) * cos_clamped(roll) + SIDE_SENSOR_OFFSET


def stable_right_distance(data, model, desired_heading):
    """Right-hand wall distance -> what the maze follower runs on"""
    return stable_side_distance(data, model, desired_heading, side=+1)


def stable_front_distance(data, model, desired_heading):
    """Forward clearance, projected onto the corridor axis"""
    raw = read_range(data, model, "front_distance")
    if not np.isfinite(raw):
        return np.inf
    _, pitch = get_roll_pitch(data)
    yaw_error = angle_diff(get_yaw(data), desired_heading)
    return raw * cos_clamped(yaw_error) * cos_clamped(pitch)


def condition_held(movement_state, data, key, condition, hold=None):
    """True once `condition` has stayed true for DEBOUNCE_TIME"""
    hold = DEBOUNCE_TIME if hold is None else hold
    since = movement_state["cond_since"]
    if not condition:
        since[key] = None
        return False
    if since.get(key) is None:
        since[key] = data.time
    return data.time - since[key] >= hold


def front_obstacle_detection(front_distance):
    return 0.03 < front_distance < FRONT_BLOCK_DISTANCE


def classify_obstacle(data, model, open_sky):
    """
    Decide whether the thing ahead is something to climb or something to walk around by using the front_height_sensor 
    """
    front = read_range(data, model, "front_distance")
    height_hit = read_range(data, model, "front_height_sensor")

    if not np.isfinite(front):
        return None                       # nothing ahead to classify

    if not (np.isfinite(height_hit) and height_hit < front + 1.0):
        return "box"                      # low enough to climb over

    return "wall" if open_sky else "maze"  #prevent the ant from walking around a wall due to the "keep one hand on the wall"-rule


ANGLED_SENSOR_PAIRS = (
    (55.0, "front_left_55", "front_right_55"),   # splits close in  (~1.8-2.4 m here)
    (20.0, "front_left_20", "front_right_20"),   # splits far out   (~7-9 m here)
)


def choose_go_around_side(data, model, front_distance, yaw_error):
    """
    Shorter way round the obstacle ahead: +1 left, -1 right, 0 for "no idea"
    """
    if abs(yaw_error) > ANGLED_ALIGN_TOLERANCE:
        return 0                          # too crooked for the two rays to be comparable
    for angle_deg, left_name, right_name in ANGLED_SENSOR_PAIRS:
        limit = front_distance / np.cos(np.radians(angle_deg)) + ANGLED_CLEAR_MARGIN
        left = read_range(data, model, left_name)
        right = read_range(data, model, right_name)
        left_clear = (not np.isfinite(left)) or left > limit
        right_clear = (not np.isfinite(right)) or right > limit
        if left_clear != right_clear:
            return +1 if left_clear else -1
    return 0                              # symmetric, or nothing visible yet


def lateral_offset(position, origin, heading):
    delta = np.asarray(position[:2], dtype=float) - np.asarray(origin[:2], dtype=float)
    return float(-delta[0] * np.sin(heading) + delta[1] * np.cos(heading))


# In-place rotation
LEFT_SIDE_LEGS = {name for name, leg in legs.items() if leg["hip_sign"] > 0}
RIGHT_SIDE_LEGS = set(legs) - LEFT_SIDE_LEGS


def angle_diff(a, b):
    d = a - b
    return (d + np.pi) % (2 * np.pi) - np.pi


def snap_to_cardinal(heading):
    #Round a heading to the nearest multiple of 90 degrees
    return round(heading / (np.pi / 2)) * (np.pi / 2)


# distance tracking, used to time the corner manoeuvres
def start_progress_tracking(data, movement_state):
    movement_state["anchor"] = data.qpos[:2].copy()


def distance_traveled(data, movement_state):
    return float(np.linalg.norm(data.qpos[:2] - movement_state["anchor"]))


# state-machine bookkeeping 
def begin_mode(movement_state, data, mode):
    """Enter a mode, resetting its distance anchor and its debounce timers"""
    movement_state["mode"] = mode
    movement_state["mode_start"] = data.time
    movement_state["cond_since"] = {}
    start_progress_tracking(data, movement_state)


def start_pivot(movement_state, data, direction, next_mode="maze_reacquire_right_wall"):
    """Commit to the next cardinal heading, +1 for left, -1 for right"""
    movement_state["pivot_target"] = snap_to_cardinal(movement_state["desired_heading"] + direction * (np.pi / 2))
    movement_state["desired_heading"] = movement_state["pivot_target"]
    movement_state["pivot_reached_at"] = None
    movement_state["after_pivot"] = next_mode
    begin_mode(movement_state, data, "turn_left" if direction > 0 else "turn_right")


def start_backup(movement_state, data, resume_mode):
    """Wedged against something: run the gait in reverse to buy room, then retry"""
    movement_state["resume_after_backup"] = resume_mode
    begin_mode(movement_state, data, "back_up")


def start_realign(movement_state, data, target_heading, next_mode):
    """Turn to face an absolute heading, then hand on to `next_mode`"""
    movement_state["pivot_target"] = snap_to_cardinal(target_heading)
    movement_state["desired_heading"] = movement_state["pivot_target"]
    movement_state["pivot_reached_at"] = None
    movement_state["after_pivot"] = next_mode
    begin_mode(movement_state, data, "realign")


# Controller
The next section contains the various controller used for the walking movement, especially the CPG Control logic.

In [9]:
def pd_control(desired_pos, current_pos, current_vel, kp, kd):
    error = desired_pos - current_pos
    command = kp * error - kd * current_vel
    return np.clip(command, -1.0, 1.0)


def apply_leg_pd(ctrl, data, leg_name, desired_hip, desired_ankle):
    """Small shared helper so every controller below applies PD the same way"""
    leg = legs[leg_name]
    hip_actuator, ankle_actuator = leg["actuators"]
    hip_qpos, ankle_qpos = leg["qpos"]
    hip_qvel, ankle_qvel = leg["qvel"]

    ctrl[hip_actuator] = pd_control(desired_hip, data.qpos[hip_qpos], data.qvel[hip_qvel], KP_HIP, KD_HIP)
    ctrl[ankle_actuator] = pd_control(desired_ankle, data.qpos[ankle_qpos], data.qvel[ankle_qvel], KP_ANKLE, KD_ANKLE)


def neutral_stance_controller(model, data):  #keeps all legs in a neutral stance
    ctrl = np.zeros(model.nu)
    for leg_name, leg in legs.items():
        desired_hip, desired_ankle = neutral_stance_target(leg_name, leg)
        apply_leg_pd(ctrl, data, leg_name, desired_hip, desired_ankle)
    return ctrl

#the following functions were generated for the climbing controller -> not working properly yet, so they are commented out for now

"""
def brace_controller(model, data, movement_state):
    ctrl = np.zeros(model.nu)
    for leg_name, leg in legs.items():
        if leg_name in movement_state["locked_joint_targets"]:
            desired_hip, desired_ankle = movement_state["locked_joint_targets"][leg_name]
        else:
            desired_hip, desired_ankle = neutral_stance_target(leg_name, leg)
        apply_leg_pd(ctrl, data, leg_name, desired_hip, desired_ankle)
    return ctrl


def probe_leg_controller(t, model, data, movement_state):
    leg_name = movement_state["active_leg"]
    leg = legs[leg_name]
    ctrl = brace_controller(model, data, movement_state)  # everyone else stays braced/planted

    elapsed = t - movement_state["mode_start"]
    duration = 3.0
    progress = np.clip(elapsed / duration, 0.0, 1.0)

    hip_scale = leg.get("hip_scale", 1.0)
    ankle_scale = leg.get("ankle_scale", 1.0)
    neutral_hip, neutral_ankle = neutral_stance_target(leg_name, leg)

    hip_forward = 0.45 * WALK_DIRECTION * leg["hip_sign"] * hip_scale
    ankle_folded = neutral_ankle * 0.55     # partial fold, well inside range
    ankle_extended = neutral_ankle * 1.15   # partial extend, still inside range

    if progress < 0.35:
        p = progress / 0.35
        desired_hip = (1 - p) * neutral_hip + p * hip_forward
        desired_ankle = (1 - p) * neutral_ankle + p * ankle_folded
    else:
        p = (progress - 0.35) / 0.65
        desired_hip = hip_forward
        desired_ankle = (1 - p) * ankle_folded + p * ankle_extended

    apply_leg_pd(ctrl, data, leg_name, desired_hip, desired_ankle)

    if foot_is_elevated(data, model, leg_name, movement_state):
        foot_z = get_foot_height(data, model, leg_name)
        print(f"[plant] {leg_name} landed at z={foot_z:.3f}")
        lock_leg_target(data, model, movement_state, leg_name)
        movement_state["planted_legs"].add(leg_name)
        if movement_state["estimated_obstacle_height"] is None:
            movement_state["estimated_obstacle_height"] = foot_z

    return ctrl


def advance_body_controller(model, data, movement_state, cpg_state):
    ctrl = cpg_controller(model, data, cpg_state)
    for leg_name in movement_state["planted_legs"]:
        desired_hip, desired_ankle = movement_state["locked_joint_targets"][leg_name]
        apply_leg_pd(ctrl, data, leg_name, desired_hip, desired_ankle)
    return ctrl

"""

# walking and steering controller

def leg_targets_from_phase(leg_name, leg, phase, local_direction):
    """Hip and ankle targets for one leg at one point in its gait cycle"""
    hip_scale = leg.get("hip_scale", 1.0)
    ankle_scale = leg.get("ankle_scale", 1.0)

    if phase < SWING_END:
        p = phase / SWING_END
        desired_hip = ((1 - p) * (local_direction * HIP_BACK * leg["hip_sign"]) + p * (local_direction * HIP_FRONT * leg["hip_sign"])) * hip_scale
        lift_target = ANKLE_LIFT_BACK if leg_name in ["back_left", "back_right"] else ANKLE_LIFT_FRONT
        desired_ankle = lift_target * leg["ankle_sign"] * ankle_scale
    elif phase < HOLD_END:
        desired_hip = local_direction * HIP_FRONT * leg["hip_sign"] * hip_scale
        desired_ankle = ANKLE_SUPPORT * leg["ankle_sign"] * ankle_scale
    else:
        p = (phase - HOLD_END) / (1.0 - HOLD_END)
        desired_hip = ((1 - p) * (local_direction * HIP_FRONT * leg["hip_sign"]) + p * (local_direction * HIP_BACK * leg["hip_sign"])) * hip_scale
        desired_ankle = ANKLE_SUPPORT * leg["ankle_sign"] * ankle_scale

    return desired_hip, desired_ankle


def cpg_controller(model, data, cpg_state, steer=0.0, direction=1.0):
    """
    Generate hip and ankle targets from four coupled CPG phases
    """
    steer = float(np.clip(steer, -1.0, 1.0))
    inner_direction = 1.0 - 2.0 * abs(steer) #1.0 for straight, 0.0 for full steer, negative for reverse

    ctrl = np.zeros(model.nu)

    for leg_name, leg in legs.items():
        on_right_side = leg_name in RIGHT_SIDE_LEGS
        if steer > 0:
            local_direction = inner_direction if on_right_side else 1.0
        elif steer < 0:
            local_direction = 1.0 if on_right_side else inner_direction
        else:
            local_direction = 1.0

        phase = cpg_state["phases"][leg_name] / (2.0 * np.pi)
        desired_hip, desired_ankle = leg_targets_from_phase(leg_name, leg, phase, WALK_DIRECTION * local_direction * direction)
        apply_leg_pd(ctrl, data, leg_name, desired_hip, desired_ankle)

    return ctrl


def rotate_left_controller(model, data, cpg_state):
    """Pivot left in place"""
    return cpg_controller(model, data, cpg_state, steer=+1.0)


def rotate_right_controller(model, data, cpg_state):
    """Pivot right in place"""
    return cpg_controller(model, data, cpg_state, steer=-1.0)


def heading_steer(data, target_heading, max_steer=STEER_MAX_FOLLOW):
    """Proportional heading hold, offset past the measured steering dead zone"""
    error = angle_diff(target_heading, get_yaw(data))
    if abs(error) < HEADING_TOLERANCE:
        return 0.0
    magnitude = STEER_DEADBAND + STEER_GAIN * (abs(error) - HEADING_TOLERANCE)
    return float(np.sign(error) * np.clip(magnitude, 0.0, max_steer))


def wall_following_steer(data, side_distance, desired_heading, side=+1):
    """Heading hold, biased by how far the ant is from the wall it is following."""
    wall_error = np.clip(side_distance - WALL_FOLLOW_TARGET, -1.0, 1.0)
    heading_offset = np.clip(WALL_STEER_GAIN * wall_error, -WALL_STEER_MAX, WALL_STEER_MAX)
    return heading_steer(data, desired_heading - side * heading_offset)


def lane_steer(data, target_y, heading):
    """Heading hold, biased to bring the ant back onto a lane (a target y)"""
    lane_error = float(np.clip(data.qpos[1] - target_y, -2.5, 2.5))
    heading_offset = np.clip(LANE_STEER_GAIN * lane_error, -LANE_STEER_MAX, LANE_STEER_MAX)
    return heading_steer(data, heading - heading_offset)


# Kinemantics Foot Position Testing

The function scan_front_left_workspace is a helper function that tests the kinematics of the front-left leg without simulating forces. It generates a copy of the simulation state so that we could test the leg without changing the overall simulation. We only needed to do the simulation for one leg, as all legs are configured equally and just mirrored to their respective positions in relation to the torso. 

The function finds the joint and ankle indices for the front left leg as well as the foot site used to track the foot's position. The current torso position is stored, so the foot position can be calculated in relation to the torso. A list of possible joint relations is calculated by sampling hip and ankle values across a defined range and then sorted by height and forward reach. This let us identify the most promising configurations to use for foot positioning. 

In [10]:
def scan_front_left_workspace(model, data):
    test_data = mujoco.MjData(model)

    hip_qpos = model.joint("hip_1").qposadr[0]
    ankle_qpos = model.joint("ankle_1").qposadr[0]

    foot_site_id = mujoco.mj_name2id(
        model,
        mujoco.mjtObj.mjOBJ_SITE,
        "foot_1_touch_site",
    )

    torso_x = data.qpos[0]
    torso_y = data.qpos[1]
    torso_z = data.qpos[2]

    candidates = []

    hip_values = np.linspace(-0.52, 0.52, 9)
    ankle_values = np.linspace(0.52, 1.22, 9)

    for hip in hip_values:
        for ankle in ankle_values:
            test_data.qpos[:] = data.qpos[:]
            test_data.qvel[:] = 0.0

            test_data.qpos[hip_qpos] = hip
            test_data.qpos[ankle_qpos] = ankle

            mujoco.mj_forward(model, test_data)

            foot_pos = test_data.site_xpos[foot_site_id].copy()

            foot_x_rel = foot_pos[0] - torso_x
            foot_y_rel = foot_pos[1] - torso_y
            foot_z = foot_pos[2]

            candidates.append(
                {
                    "hip": hip,
                    "ankle": ankle,
                    "foot_x_rel": foot_x_rel,
                    "foot_y_rel": foot_y_rel,
                    "foot_z": foot_z,
                }
            )

    # Sort by height first, then forward reach.
    candidates = sorted(
        candidates,
        key=lambda c: (c["foot_z"], c["foot_x_rel"]),
        reverse=True,
    )

    print("Best front-left kinematic poses:")
    for c in candidates[:10]:
        print(
            f"hip={c['hip']:.3f}, "
            f"ankle={c['ankle']:.3f}, "
            f"x_rel={c['foot_x_rel']:.3f}, "
            f"y_rel={c['foot_y_rel']:.3f}, "
            f"z={c['foot_z']:.3f}"
        )

# Behaviors 

One behaviour per kind of obstacle, all sharing the same primitives (heading,hold, wall following, pivots, backing out). Our function run_simulation() just senses, picks
the behaviour that owns the current mode and gait configurations. So adding the next obstacle type means adding a function here, not editing the run_simulation() function.

walk: cruise on the home heading, classify what turns up

turn_left / turn_right: shared pivot, hands back via movement_state["after_pivot"]

back_up: shared unwedging

maze_*: right-hand wall following (the labyrinth)

wall_*: go round a free-standing obstacle and rejoin course

brace / probe / advance: climb over a low box


In [11]:
def sense_environment(model, data, movement_state):
    """The place every rangefinder is read and conditioned"""
    heading = movement_state["desired_heading"]

    for side, key in ((+1, "right_belief"), (-1, "left_belief")):
        value = stable_side_distance(data, model, heading, side)
        if value is not None:
            movement_state[key] = value

    raw_left = read_range(data, model, "left_side_distance")
    raw_right = read_range(data, model, "right_side_distance")
    clear_now = float(raw_left > OPEN_SKY_DISTANCE and raw_right > OPEN_SKY_DISTANCE)
    alpha = model.opt.timestep / OPEN_SKY_WINDOW
    movement_state["open_sky_level"] += alpha * (clear_now - movement_state["open_sky_level"])
    if movement_state["open_sky_level"] > OPEN_SKY_FRACTION:
        movement_state["open_sky"] = True
    elif movement_state["open_sky_level"] < OPEN_SKY_RELEASE:
        movement_state["open_sky"] = False
    open_sky = movement_state["open_sky"]

    _, pitch_now = get_roll_pitch(data)
    slope_alpha = model.opt.timestep / TERRAIN_SLOPE_WINDOW
    movement_state["slope_level"] += slope_alpha * (-np.degrees(pitch_now) - movement_state["slope_level"])

    front = stable_front_distance(data, model, heading)

    if open_sky and front < WALL_SCAN_DISTANCE and movement_state["go_around_side"] == 0:
        vote = choose_go_around_side(data, model, front, angle_diff(get_yaw(data), heading))
        # Latch only an answer that survives a moment, so one lucky frame between leg swings cannot decide which way the ant walks round the obstacle
        if vote != 0 and condition_held(movement_state, data, f"side{vote}", True, hold=ANGLED_AGREE_TIME):
            movement_state["go_around_side"] = vote

    return {
        "heading": heading,
        "front": front,
        "right": movement_state["right_belief"],
        "left": movement_state["left_belief"],
        "open_sky": open_sky,
        "slope": movement_state["slope_level"],
    }


def hold_heading(model, data, cpg_state, heading):
    return cpg_controller(model, data, cpg_state, steer=heading_steer(data, heading))


#cruising

def walk_step(model, data, cpg_state, movement_state, sense):
    """Cruise on the committed heading until something needs a decision"""
    heading = sense["heading"]
    if sense["open_sky"]:
        u = cpg_controller(model, data, cpg_state, steer=lane_steer(data, movement_state["home_y"], heading))
    else:
        u = hold_heading(model, data, cpg_state, heading)

    facing_intended_way = abs(angle_diff(heading, get_yaw(data))) < ALIGN_TOLERANCE

    if facing_intended_way and condition_held(movement_state, data, "front", front_obstacle_detection(sense["front"])):
        kind = classify_obstacle(data, model, sense["open_sky"])
        movement_state["obstacle_type"] = kind
        print(f"{data.time:7.2f}  obstacle ahead -> classified as {kind}")
        if kind == "box":
            begin_mode(movement_state, data, "brace")
        elif kind == "wall":
            start_wall_detour(model, data, movement_state)
        else:
            # First wall of a labyrinth: pivot left so it ends up on the ant's right, then follow it
            start_pivot(movement_state, data, +1, "maze_reacquire_right_wall")

    elif sense["open_sky"] and condition_held(movement_state, data, "slope", sense["slope"] > TERRAIN_ENTER_SLOPE, hold=TERRAIN_ENTER_HOLD):
        print(f"{data.time:7.2f}  ground rising ({sense['slope']:.1f} deg) -> crossing terrain")
        movement_state["desired_heading"] = WALK_HOME_HEADING
        begin_mode(movement_state, data, "terrain_cross")

    """ 
    Nothing else switches out of walk. An earlier version also started following any wall that turned up on the right, without a front obstacle to justify it. That
    made the ant grab the *outside* of the labyrinth on its way out and follow it back down, so the rule now is simply: react to what is in front of you.
    """
    return u


#shared pieces
def pivot_step(model, data, cpg_state, movement_state, sense):
    """Turn until the yaw actually reaches the target cardinal heading."""
    heading_error = angle_diff(movement_state["pivot_target"], get_yaw(data))
    steer = np.sign(heading_error) if abs(heading_error) > PIVOT_TOLERANCE else 0.0
    u = cpg_controller(model, data, cpg_state, steer=steer)

    if abs(heading_error) <= PIVOT_TOLERANCE:
        if movement_state["pivot_reached_at"] is None:
            movement_state["pivot_reached_at"] = data.time
        elif data.time - movement_state["pivot_reached_at"] >= PIVOT_SETTLE_TIME:
            begin_mode(movement_state, data, movement_state["after_pivot"])
    else:
        movement_state["pivot_reached_at"] = None      # fell back off target

    timeout = REALIGN_TIMEOUT if movement_state["mode"] == "realign" else PIVOT_TIMEOUT
    if data.time - movement_state["mode_start"] > timeout:
        # Not turning at all means the body is jammed, not that the turn is slow
        movement_state["pivot_reached_at"] = None
        start_backup(movement_state, data, movement_state["mode"])

    return u


def backup_step(model, data, cpg_state, movement_state, sense):
    u = cpg_controller(model, data, cpg_state, steer=0.0, direction=-1.0)
    if data.time - movement_state["mode_start"] > BACKUP_TIME:
        begin_mode(movement_state, data, movement_state["resume_after_backup"])
    return u


def unstick_check(model, data, movement_state, window=None, distance=None):
    """
    Walking but not moving: something is caught where no ray can see it
    Returns True if a recovery was started, so the caller should stop what it was doing.
    """
    window = STUCK_WINDOW if window is None else window
    distance = STUCK_DISTANCE if distance is None else distance
    if np.linalg.norm(data.qpos[:2] - movement_state["prog_anchor"]) > distance:
        movement_state["prog_anchor"] = data.qpos[:2].copy()
        movement_state["prog_time"] = data.time
        return False
    if data.time - movement_state["prog_time"] > window:
        movement_state["prog_anchor"] = data.qpos[:2].copy()
        movement_state["prog_time"] = data.time
        start_backup(movement_state, data, movement_state["mode"])
        return True
    return False


# the labyrinth
def maze_navigation_step(model, data, cpg_state, movement_state, sense):
    """Right-hand wall following: keep a wall on the right and never let go of it"""
    heading = sense["heading"]
    mode = movement_state["mode"]

    if sense["open_sky"]:
        """
        There is no corridor left to follow, so the labyrinth is behind us. Hand back to cruising on the homeheading rather than following the outside of the maze forever.
        """
        print(f"{data.time:7.2f}  open on both sides -> out of the labyrinth " f"at x={data.qpos[0]:.2f}, y={data.qpos[1]:.2f}")
        movement_state["desired_heading"] = WALK_HOME_HEADING
        begin_mode(movement_state, data, "walk")
        return hold_heading(model, data, cpg_state, WALK_HOME_HEADING)

    if unstick_check(model, data, movement_state):
        return backup_step(model, data, cpg_state, movement_state, sense)

    blocked = condition_held(movement_state, data, "front", sense["front"] < FRONT_BLOCK_DISTANCE)
    wall_seen = condition_held(movement_state, data, "seen", sense["right"] < WALL_LOST_DISTANCE)
    wall_gone = condition_held(movement_state, data, "gone", sense["right"] > WALL_LOST_DISTANCE)

    if mode == "maze_follow_right_wall":
        u = cpg_controller(model, data, cpg_state, steer=wall_following_steer(data, sense["right"], heading, +1))
        if blocked:
            # Wall still on the right and now blocked ahead too: inside corner, nowhere to go but left
            start_pivot(movement_state, data, +1, "maze_reacquire_right_wall")
        elif wall_gone:
            # The wall on the right ended. Don't pivot yet -- walk on far enough that the right-hand legs clear the corner before turning into it
            begin_mode(movement_state, data, "maze_clear_before_turn_right")

    elif mode == "maze_clear_before_turn_right":
        u = hold_heading(model, data, cpg_state, heading)
        travelled = distance_traveled(data, movement_state)
        opening_up = (movement_state["open_sky_level"] > OPEN_SKY_PENDING and travelled < 3.0 * CORNER_CLEAR_DISTANCE)
        if blocked:
            start_pivot(movement_state, data, +1, "maze_reacquire_right_wall")
        elif travelled > CORNER_CLEAR_DISTANCE and not opening_up:
            start_pivot(movement_state, data, -1, "maze_reacquire_right_wall")

    else:   # maze_reacquire_right_wall
        u = hold_heading(model, data, cpg_state, heading)
        if blocked:
            start_pivot(movement_state, data, +1, "maze_reacquire_right_wall")
        elif wall_seen:
            begin_mode(movement_state, data, "maze_follow_right_wall")
        elif distance_traveled(data, movement_state) > REACQUIRE_MAX_DISTANCE:
            # Walked well past where the wall should have been: a wide opening rather than a corner, so keep turning right around it
            start_pivot(movement_state, data, -1, "maze_reacquire_right_wall")

    return u


#free-standing wall
def start_wall_detour(model, data, movement_state):
    """Commit to walking around a free-standing obstacle"""
    side = movement_state["go_around_side"] or +1     # nothing seen either way -> left
    movement_state["wall_side"] = side                # obstacle ends up on this hand
    movement_state["wall_corners_turned"] = 0
    movement_state["wall_start_pos"] = data.qpos[:2].copy()
    movement_state["wall_start_heading"] = movement_state["desired_heading"]
    print(f"{data.time:7.2f}  free-standing wall -> passing it on the " f"{'left' if side > 0 else 'right'}")
    start_pivot(movement_state, data, side, "wall_reacquire")


def wall_navigation_step(model, data, cpg_state, movement_state, sense):
    """Round a free-standing obstacle, then rejoin the original course"""
    heading = sense["heading"]
    side = movement_state["wall_side"]
    side_distance = sense["right"] if side > 0 else sense["left"]
    mode = movement_state["mode"]

    # Past the second corner the ant is on the far face, heading back toward the course
    if movement_state["wall_corners_turned"] >= 2:
        offset = lateral_offset(data.qpos[:2], movement_state["wall_start_pos"], movement_state["wall_start_heading"])
        if abs(offset) < WALL_RETURN_TOLERANCE:
            print(f"{data.time:7.2f}  back on the original line at " f"x={data.qpos[0]:.2f}, y={data.qpos[1]:.2f} -> obstacle cleared")
            movement_state["go_around_side"] = 0
            # Turn back to the original heading before carrying on
            start_realign(movement_state, data, movement_state["wall_start_heading"], "walk")
            return hold_heading(model, data, cpg_state, movement_state["desired_heading"])

    if unstick_check(model, data, movement_state):
        return backup_step(model, data, cpg_state, movement_state, sense)

    blocked = condition_held(movement_state, data, "front", sense["front"] < FRONT_BLOCK_DISTANCE)
    wall_seen = condition_held(movement_state, data, "seen", side_distance < WALL_SIDE_LOST)
    wall_gone = condition_held(movement_state, data, "gone", side_distance > WALL_SIDE_LOST, hold=WALL_GONE_HOLD)

    if distance_traveled(data, movement_state) > WALL_MAX_LEG_DISTANCE:
        #Fall back to the general-purpose behaviour
        print(f"{data.time:7.2f}  [warning] face longer than {WALL_MAX_LEG_DISTANCE} m " f"-> not free-standing after all, reverting to wall following")
        begin_mode(movement_state, data, "maze_follow_right_wall")
        return hold_heading(model, data, cpg_state, heading)

    if mode == "wall_follow":
        u = cpg_controller(model, data, cpg_state, steer=wall_following_steer(data, side_distance, heading, side))
        if blocked:
            start_pivot(movement_state, data, side, "wall_reacquire")   # turn away from it
        elif wall_gone:
            begin_mode(movement_state, data, "wall_clear")

    elif mode == "wall_clear":
        u = hold_heading(model, data, cpg_state, heading)
        if blocked:
            start_pivot(movement_state, data, side, "wall_reacquire")
        elif distance_traveled(data, movement_state) > WALL_CORNER_CLEAR:
            movement_state["wall_corners_turned"] += 1
            start_pivot(movement_state, data, -side, "wall_reacquire")  # round the corner

    else:   # wall_reacquire
        u = hold_heading(model, data, cpg_state, heading)
        if blocked:
            start_pivot(movement_state, data, side, "wall_reacquire")
        elif wall_seen:
            begin_mode(movement_state, data, "wall_follow")
        elif distance_traveled(data, movement_state) > REACQUIRE_MAX_DISTANCE:
            movement_state["wall_corners_turned"] += 1
            start_pivot(movement_state, data, -side, "wall_reacquire")

    return u


# climbing a box -> not working
"""
def climb_step(model, data, cpg_state, movement_state, sense):
    mode = movement_state["mode"]

    if mode == "brace":
        u = neutral_stance_controller(model, data)
        if data.time - movement_state["mode_start"] > BRACE_DURATION:
            movement_state["active_leg"] = movement_state["probe_queue"].pop(0)
            begin_mode(movement_state, data, "probe")

    elif mode == "probe":
        u = probe_leg_controller(data.time, model, data, movement_state)
        if movement_state["active_leg"] in movement_state["planted_legs"]:
            if movement_state["probe_queue"]:
                begin_mode(movement_state, data, "advance")
            else:
                # all four legs planted on top -> obstacle cleared
                begin_mode(movement_state, data, "walk")
                movement_state["planted_legs"].clear()
                movement_state["locked_joint_targets"].clear()
                movement_state["probe_queue"] = list(CLIMB_SEQUENCE)

    else:   # advance
        u = advance_body_controller(model, data, movement_state, cpg_state)
        if data.time - movement_state["mode_start"] > 0.8:
            movement_state["active_leg"] = movement_state["probe_queue"].pop(0)
            begin_mode(movement_state, data, "probe")

    return u
"""

#sloped / uneven ground
def terrain_step(model, data, cpg_state, movement_state, sense):
    """
    Cross rising or uneven ground, holding the home heading

    Deliberately almost empty: the adaptation is the slower gait cycle, which run_simulation selects from the mode, plus keeping the nose pointed at
    WALK_HOME_HEADING so the ant tracks straight instead of wandering downhill.
    """
    if unstick_check(model, data, movement_state, window=TERRAIN_STUCK_WINDOW, distance=TERRAIN_STUCK_DISTANCE):
        return backup_step(model, data, cpg_state, movement_state, sense)
    return cpg_controller(model, data, cpg_state, steer=lane_steer(data, movement_state["home_y"], WALK_HOME_HEADING))


BRACE_DURATION = 0.6

# Which behaviour owns which mode. run_simulation does nothing but look this up
BEHAVIOURS = [
    (lambda m: m == "walk", walk_step),
    (lambda m: m in ("turn_left", "turn_right", "realign"), pivot_step),
    (lambda m: m == "back_up", backup_step),
    (lambda m: m.startswith("maze_"), maze_navigation_step),
    (lambda m: m.startswith("wall_"), wall_navigation_step),
    (lambda m: m.startswith("terrain_"), terrain_step),
    #(lambda m: m in ("brace", "probe", "advance"), climb_step),
]


def behaviour_for(mode):
    for matches, step in BEHAVIOURS:
        if matches(mode):
            return step
    raise KeyError(f"no behaviour owns mode {mode!r}")


# Simulation Setup and Control Loop

Lastly, we define the helper functions used to setting up the simulation environment and rendering the robot. 

In [12]:
# Simulation helper


def make_camera():
    """
    Use this method to change the camera settings for the video
    """                          
    camera = mujoco.MjvCamera()
    mujoco.mjv_defaultCamera(camera)

    camera.distance = 20.0
    camera.azimuth = 90
    camera.elevation = -20
    camera.lookat[:] = [0.0, 0.0, 0.5]

    return camera

def update_camera(camera, data):
    """
    Use this method to change the camera settings for the video
    """
    camera.lookat[:] = data.qpos[:3]
    camera.distance = 15.0
    camera.azimuth = 90
    camera.elevation = -20

def get_sensor(data, model, name):
    sid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SENSOR, name)
    adr = model.sensor_adr[sid]
    dim = model.sensor_dim[sid]
    return data.sensordata[adr:adr + dim]


def run_simulation(duration=30.0, capture_fps=5, playback_fps=30, show=True, verbose=True):
    """
    Drive the ant and record the run.

    duration: how long does the simulation run
    capture_fps and playback_fps: used to minimize the mp4 file size and speed up the video, normal speed can be regained by setting capture_fps to the same value as playback_fps (e.g. both 30)
    """
    data = mujoco.MjData(model)
    mujoco.mj_forward(model, data)
    renderer = mujoco.Renderer(model, height=480, width=640)

    frames = []
    ctrl_log = []
    camera = make_camera()

    start_x, start_y = data.qpos[0], data.qpos[1]
    movement_state = make_movement_state()
    cpg_state = make_cpg_state()
    # Measure resting foot height once from the start pose to use as ground height -> not working
    #movement_state["ground_foot_z"] = np.mean([get_foot_height(data, model, l) for l in legs])
    
    movement_state["desired_heading"] = snap_to_cardinal(get_yaw(data))
    movement_state["prog_anchor"] = data.qpos[:2].copy()
    # The lane the ant should return to whenever it is out in the open
    movement_state["home_y"] = float(data.qpos[1])
    begin_mode(movement_state, data, "walk")

    previous_mode = None

    while data.time < duration:
        # The gait cycle is the one thing terrain changes, so the runner picks it from the mode rather than every behaviour having to thread it through
        gait_frequency = (TERRAIN_FREQUENCY if movement_state["mode"].startswith("terrain_") else FREQUENCY)
        update_cpg_phases(cpg_state, dt=model.opt.timestep, frequency=gait_frequency, coupling_strength=CPG_COUPLING_STRENGTH)

        sense = sense_environment(model, data, movement_state)
        u = behaviour_for(movement_state["mode"])(model, data, cpg_state, movement_state, sense)

        if verbose and movement_state["mode"] != previous_mode:         # print relevant variables whenever the mode of movement changes
            print(f"{data.time:7.2f}  {movement_state['mode']:28s} "
                  f"heading={np.degrees(movement_state['desired_heading']) % 360:3.0f} deg  "
                  f"pos=({data.qpos[0]:6.2f},{data.qpos[1]:6.2f})  "
                  f"front={min(sense['front'], 99):5.2f}  "
                  f"L={min(sense['left'], 99):5.2f} R={min(sense['right'], 99):5.2f}")
            previous_mode = movement_state["mode"]

        data.ctrl[:] = u
        ctrl_log.append(u.copy())
        mujoco.mj_step(model, data)

        if len(frames) < data.time * capture_fps:
            update_camera(camera,data)
            renderer.update_scene(data, camera=camera)
            frames.append(renderer.render())

    end_x = data.qpos[0]
    end_y = data.qpos[1]

    ctrl_log = np.array(ctrl_log)           # store ctrl values

    print()
    print("dx:", end_x - start_x)
    print("dy:", end_y - start_y)
    print("distance:", np.sqrt((end_x - start_x)**2 + (end_y - start_y)**2))
    print("final height:", data.qpos[2])

    print("ctrl min:", np.round(ctrl_log.min(axis=0), 2))
    print("ctrl max:", np.round(ctrl_log.max(axis=0), 2))
    print("ctrl mean abs:", np.round(np.mean(np.abs(ctrl_log), axis=0), 2))
    print("saturation fraction:", np.mean(np.abs(ctrl_log) > 0.95))

    if show:
        media.show_video(frames, fps=playback_fps)

    return {
        "dx": end_x - start_x,
        "dy": end_y - start_y,
        "distance": np.sqrt((end_x - start_x)**2 + (end_y - start_y)**2),
        "final_height": data.qpos[2],
        "saturation_fraction": np.mean(np.abs(ctrl_log) > 0.95),
    }, frames


In [ ]:
# Run the controller
# Captured at 3 fps and played at 30, so the video runs at 10x real time. To watch the full obstacle set duration to 360 seconds or more, and set capture_fps=30 to see it in real time
metrics, frames = run_simulation(duration=15.0, capture_fps=3, playback_fps=30, show=True)


   0.00  walk                         heading=  0 deg  pos=(  0.00,  0.00)  front= 1.15  L= 7.85 R= 7.85


   0.20  obstacle ahead -> classified as maze
   0.20  turn_left                    heading= 90 deg  pos=( -0.00, -0.01)  front= 1.16  L= 7.87 R= 7.83
   4.96  maze_reacquire_right_wall    heading= 90 deg  pos=(  0.77, -0.28)  front= 7.43  L= 2.63 R= 1.07
   5.18  maze_follow_right_wall       heading= 90 deg  pos=(  0.77, -0.26)  front= 7.43  L= 2.64 R= 1.06

dx: 0.3371164414248012
dy: 1.419220026363445
distance: 1.4587093536102311
final height: 0.4872251585445924
ctrl min: [-1. -1. -1. -1. -1. -1. -1. -1.]
ctrl max: [1. 1. 1. 1. 1. 1. 1. 1.]
ctrl mean abs: [0.04 0.2  0.04 0.11 0.04 0.12 0.04 0.2 ]
saturation fraction: 0.04938374417055297


This browser does not support the video tag.
